In [0]:
# 1) Configuración y lectura de la capa BRONZE
from pyspark.sql import functions as F

SCHEMA = "credito_riesgo_lab"
TABLA_BRONZE = f"workspace.{SCHEMA}.clientes_bronze"
TABLA_SILVER = f"workspace.{SCHEMA}.clientes_silver"

# spark.table() lee una tabla Delta ya guardada (lazy: no lee nada hasta que hace falta)
bronze = spark.table(TABLA_BRONZE)
print(f"Bronze: {bronze.count()} filas")
bronze.printSchema()

In [0]:
# 2) Capa SILVER — datos limpios y confiables
# Reglas de limpieza (cada filter elimina filas "malas"):
#   a) dropDuplicates: quita duplicados por cliente_id (los que inyectamos en el notebook 00)
#   b) isNotNull: quita filas sin ingresos
#   c) rangos válidos: edad 18-100, endeudamiento 0-1, ltv 0-1.2
#   d) withColumn: columna derivada útil (antigüedad en años)
silver = (
    bronze
    .dropDuplicates(["cliente_id"])
    .filter(F.col("ingresos").isNotNull())
    .filter((F.col("edad") >= 18) & (F.col("edad") <= 100))
    .filter(F.col("endeudamiento").between(0, 1))
    .filter(F.col("ltv").between(0, 1.2))
    .withColumn("antiguedad_anios", F.round(F.col("antiguedad_meses") / 12, 1))
)

(
    silver.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLA_SILVER)
)

print(f"Silver: {spark.table(TABLA_SILVER).count()} filas (limpias)")

In [0]:
# 3) Capa GOLD (nivel cliente) — score de riesgo con reglas de negocio
# Puntos de riesgo (cuanto más alto, peor cliente):
#   - historial de pagos: moroso +40, atrasos_ocasionales +20
#   - endeudamiento (DTI): hasta +30 (proporcional al ratio)
#   - ltv: hasta +20 (proporcional)
#   - antigüedad < 12 meses: +10 (cliente nuevo = más incierto)
puntos_riesgo = (
    F.when(F.col("historial_pagos") == "moroso", 40)
     .when(F.col("historial_pagos") == "atrasos_ocasionales", 20)
     .otherwise(0)
    + F.round(F.col("endeudamiento") * 30)
    + F.round(F.col("ltv") * 20)
    + F.when(F.col("antiguedad_meses") < 12, 10).otherwise(0)
)

gold = (
    spark.table(TABLA_SILVER)
    .withColumn("score_riesgo", puntos_riesgo.cast("int"))
    # Umbrales del nivel: alto >= 55, medio >= 30, bajo < 30
    .withColumn("nivel_riesgo",
        F.when(F.col("score_riesgo") >= 55, F.lit("alto"))
         .when(F.col("score_riesgo") >= 30, F.lit("medio"))
         .otherwise(F.lit("bajo")))
)

TABLA_GOLD_CLIENTES = f"workspace.{SCHEMA}.clientes_gold"
(
    gold.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLA_GOLD_CLIENTES)
)

# ¿Cómo queda el reparto de clientes por nivel de riesgo?
display(gold.groupBy("nivel_riesgo").count().orderBy("nivel_riesgo"))

In [0]:
# 4) Capa GOLD (nivel negocio) — resumen por sector
# agg() permite varias agregaciones a la vez: contar, promediar, porcentaje...
resumen = (
    gold.groupBy("sector")
    .agg(
        F.count("*").alias("clientes"),
        F.round(F.avg("ingresos"), 2).alias("ingreso_medio"),
        F.round(F.avg("score_riesgo"), 1).alias("score_medio"),
        # % de clientes con riesgo alto: media de un 1/0 multiplicada por 100
        F.round(F.avg(F.when(F.col("nivel_riesgo") == "alto", 1).otherwise(0)) * 100, 1)
            .alias("pct_riesgo_alto"),
    )
    .orderBy(F.desc("pct_riesgo_alto"))
)

TABLA_GOLD_RESUMEN = f"workspace.{SCHEMA}.resumen_riesgo_sector"
(
    resumen.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLA_GOLD_RESUMEN)
)

display(resumen)

In [0]:
# 5) Verificación final: las 4 tablas del pipeline
# Regla didáctica nº 7: tras escribir una tabla, contar filas y mostrar ejemplos
for t in ["clientes_bronze", "clientes_silver", "clientes_gold", "resumen_riesgo_sector"]:
    n = spark.table(f"workspace.{SCHEMA}.{t}").count()
    print(f"workspace.{SCHEMA}.{t:<24} → {n} filas")

display(spark.table(f"workspace.{SCHEMA}.clientes_gold").limit(5))

# 01 · Pipeline Medallion — Bronze → Silver → Gold

**Objetivo:** leer `clientes_bronze` y construir las tres capas clásicas de la *arquitectura medallion*:

| Capa | Qué hace | Tabla |
| --- | --- | --- |
| **Silver** | Limpieza: deduplicar, quitar nulos, validar rangos | `clientes_silver` |
| **Gold (cliente)** | Reglas de negocio: `score_riesgo` y `nivel_riesgo` por cliente | `clientes_gold` |
| **Gold (negocio)** | Agregados por sector | `resumen_riesgo_sector` |

> Ejecuta antes `00_datos_sinteticos`.